In [4]:
# الخلية 1: الاستيراد والتحميل والمعالجة الأولية
import pandas as pd
import os # المكتبة الجديدة لمعالجة المسارات
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import joblib

# 1. تحديد اسم الملف والمحاولة الذكية للعثور عليه
DATA_FILE_NAME = 'processed.cleveland.data' 
RELATIVE_PATH = os.path.join('data', DATA_FILE_NAME)

# محاولة تحميل البيانات بالمسار النسبي أولاً
try:
    # 2. تحميل البيانات
    df = pd.read_csv(RELATIVE_PATH, header=None, sep=',')
    print(f"Data loaded successfully from {RELATIVE_PATH}! Shape: {df.shape}")

except FileNotFoundError:
    # إذا فشلت المحاولة الأولى، حاول البحث في المجلد الرئيسي (إذا كان الملف في data/ وليس في data/data/)
    # في بعض الأحيان، عند تشغيل Notebook من مسار آخر، نحتاج للعودة خطوة للخلف '..'
    # نقوم بتجربة مسار آخر أكثر مرونة
    FLEXIBLE_PATH = os.path.join('..', 'data', DATA_FILE_NAME)
    
    try:
        df = pd.read_csv(FLEXIBLE_PATH, header=None, sep=',')
        print(f"Data loaded successfully from {FLEXIBLE_PATH}! Shape: {df.shape}")
        
    except FileNotFoundError:
        print("----------------------------------------------------------------")
        print("FATAL ERROR: الملف غير موجود في كلا المسارين المتوقعين.")
        print("الرجاء التأكد يدوياً من الهيكل التالي:")
        print("1. الملف 'processed.cleveland.data' موجود داخل مجلد 'data'.")
        print("2. مجلد 'data' موجود في نفس مستوى مجلد 'notebooks'.")
        print("----------------------------------------------------------------")
        raise # إظهار الخطأ إذا فشل كلا المسارين

# 3. إضافة أسماء الأعمدة (هذا الجزء سيعمل الآن لأن التحميل نجح)
column_names = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target'
]

if df.shape[1] == len(column_names):
    df.columns = column_names
    print("Column names applied successfully.")
else:
    print(f"Error: Expected {len(column_names)} columns, but found {df.shape[1]}. Check file content.")

# 4. معالجة القيم المفقودة (التي هي علامات '?')
df = df.replace('?', pd.NA)
df.dropna(inplace=True)
print(f"Data shape after handling missing values: {df.shape}")

# 5. تحويل الأعمدة إلى نوع رقمي
cols_to_convert = ['ca', 'thal', 'target']
for col in cols_to_convert:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col])
        
# 6. تقسيم الميزات (X) والهدف (y)
X = df.drop('target', axis=1)
y = (df['target'] > 0).astype(int) 
print("Features (X) and Target (y) defined successfully. Ready for Model Training!")

Data loaded successfully from ../data/processed.cleveland.data! Shape: (303, 14)
Column names applied successfully.
Data shape after handling missing values: (297, 14)
Features (X) and Target (y) defined successfully. Ready for Model Training!


In [5]:
# الخلية 2: المعالجة المسبقة وتقسيم البيانات

# 1. تحديد أنواع الأعمدة (كما تم تعريفها في خطة المشروع)
numerical_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
# ملاحظة: تم إزالة 'ca' و 'thal' من الفئوية هنا لأننا سنستخدمها كأعمدة رقمية بعد تحويلها في الخلية الأولى
categorical_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope']

# 2. بناء محول المعالجة (ColumnTransformer)
# يقوم هذا بتطبيق StandardScaler على الأعمدة الرقمية و OneHotEncoder على الفئوية
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough' # للاحتفاظ بأي أعمدة أخرى لم يتم تحديدها (ca و thal المتبقية)
)

# 3. تقسيم بيانات التدريب والاختبار (80% تدريب و 20% اختبار)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Data split into training and testing sets. Ready for training pipeline.")

Data split into training and testing sets. Ready for training pipeline.


In [7]:
# الخلية 3: التدريب والتقييم الأساسي - (تم إضافة الاستيراد المفقود)

# 🚨 الإضافة الضرورية: استيراد مقاييس التقييم هنا مباشرة 🚨
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score 
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

# 1. بناء مسار التدريب (Pipeline)
rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('classifier', RandomForestClassifier(random_state=42))])

# 2. تدريب النموذج
print("Starting basic Random Forest training...")
rf_pipeline.fit(X_train, y_train)

# 3. التوقع والتقييم
y_pred_rf = rf_pipeline.predict(X_test)
y_proba_rf = rf_pipeline.predict_proba(X_test)[:, 1]

# 4. حساب المقاييس
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)
rf_auc = roc_auc_score(y_test, y_proba_rf)

print("\n--- Random Forest Basic Results ---")
print(f"Accuracy: {rf_accuracy:.4f} (الدقة)")
print(f"F1-Score: {rf_f1:.4f} (مقياس F1)")
print(f"AUC Score: {rf_auc:.4f} (منطقة تحت المنحنى ROC)")

Starting basic Random Forest training...

--- Random Forest Basic Results ---
Accuracy: 0.9000 (الدقة)
F1-Score: 0.8750 (مقياس F1)
AUC Score: 0.9699 (منطقة تحت المنحنى ROC)


In [10]:
# الخلية 4: تحسين النموذج وحفظه - (تم إضافة الاستيراد المفقود)

# 🚨 الإضافة الضرورية: استيراد GridSearchCV هنا مباشرة 🚨
from sklearn.model_selection import GridSearchCV
import joblib 

# 1. تحديد المعاملات الفائقة للبحث
param_grid = {
    'classifier__n_estimators': [100, 200], # عدد الأشجار
    'classifier__max_depth': [10, 20, None], # أقصى عمق للشجرة
}

# 2. إعداد GridSearchCV
# NOTE: rf_pipeline مفترض أنها مُعرَّفة في الخلية 3
grid_search = GridSearchCV(rf_pipeline, param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=1)

# 3. تشغيل البحث
print("\nStarting Hyperparameter Tuning... (May take a moment)")
grid_search.fit(X_train, y_train)

# 4. استخراج النموذج الأفضل
best_model = grid_search.best_estimator_
print("\n--- Optimized Model Results ---")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best F1 Score from Grid Search: {grid_search.best_score_:.4f}")

# 5. حفظ النموذج الأفضل
# تأكد من أن مجلد 'models' موجود في المجلد الرئيسي لمشروعك
try:
    # 🚨 FIX: تغيير المسار إلى '../models/' لضمان الوصول للمجلد الرئيسي 🚨
    joblib.dump(best_model, '../models/final_model.pkl')
    print("\nModel successfully exported as models/final_model.pkl for deployment.")
except Exception as e:
    print(f"ERROR: Could not save model. Ensure 'models' folder exists. Details: {e}")


Starting Hyperparameter Tuning... (May take a moment)
Fitting 5 folds for each of 6 candidates, totalling 30 fits

--- Optimized Model Results ---
Best Parameters: {'classifier__max_depth': 10, 'classifier__n_estimators': 200}
Best F1 Score from Grid Search: 0.7804

Model successfully exported as models/final_model.pkl for deployment.
